# ELECTRA Hybrid - Sentiment Analysis
## TweetEval Dataset (3-Class: Negative / Neutral / Positive)

In [ ]:
import os, re, random, time, math, html, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import emoji

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(8, os.cpu_count() or 4))

DATA = os.path.join(os.getcwd(), "data", "splits")
PROJECT = os.getcwd()

PAD_ID, UNK_ID, CLS_ID = 0, 1, 2
MAX_VOCAB, MAX_SEQLEN, NUM_CLASSES = 30_000, 64, 3
BATCH, EPOCHS, PATIENCE = 64, 10, 4
HIDDEN, NUM_HEADS, NUM_LAYERS, FF_DIM = 128, 4, 3, 256
DROPOUT, PEAK_LR, MIN_LR = 0.15, 5e-4, 1e-6
LABEL_SMOOTH = 0.1

# --- Focal Loss & Oversampling ---
USE_FOCAL = True
FOCAL_GAMMA = 2.0
USE_OVERSAMPLING = True

print(f"Config: HIDDEN={HIDDEN}, HEADS={NUM_HEADS}, LAYERS={NUM_LAYERS}, FF={FF_DIM}")
print(f"DROPOUT={DROPOUT}, LR={PEAK_LR}, EPOCHS={EPOCHS}, LABEL_SMOOTH={LABEL_SMOOTH}")
print(f"Focal={USE_FOCAL} (gamma={FOCAL_GAMMA}), Oversampling={USE_OVERSAMPLING}")

Config: HIDDEN=128, HEADS=4, LAYERS=3, FF=256
DROPOUT=0.15, LR=0.0005, EPOCHS=10, LABEL_SMOOTH=0.1


## 1. Data Preprocessing

In [11]:
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.I)
MENTION_RE = re.compile(r"(?<!\w)@[A-Za-z0-9_]+")
HASHTAG_RE = re.compile(r"#([A-Za-z0-9_]+)")

def normalize_tweet(text):
    if pd.isna(text): return ""
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = URL_RE.sub(" <url> ", text)
    text = MENTION_RE.sub(" <user> ", text)
    text = HASHTAG_RE.sub(lambda m: f" <hashtag> {m.group(1).replace('_', ' ')} ", text)
    text = emoji.demojize(text, delimiters=(" emoji_", " "))
    text = re.sub(r"emoji_([A-Za-z0-9_+\-]+)", lambda m: "emoji_" + m.group(1).replace("-", "_"), text)
    text = re.sub(r"!{2,}", " <exclaim> ", text)
    text = re.sub(r"\?{2,}", " <question> ", text)
    text = re.sub(r"\.{3,}", " <ellipsis> ", text)
    text = re.sub(r"[^\w\s'<>\-]", " ", text, flags=re.UNICODE).replace("-", " ").lower()
    return re.sub(r"\s+", " ", text).strip()

tr_df = pd.read_csv(os.path.join(DATA, 'train.csv')).dropna(subset=['text', 'label'])
va_df = pd.read_csv(os.path.join(DATA, 'validation.csv')).dropna(subset=['text', 'label'])
te_df = pd.read_csv(os.path.join(DATA, 'test.csv')).dropna(subset=['text', 'label'])

tr_norm = [normalize_tweet(t) for t in tr_df['text']]
va_norm = [normalize_tweet(t) for t in va_df['text']]
te_norm = [normalize_tweet(t) for t in te_df['text']]

word_counts = {}
for text in tr_norm:
    for w in text.split(): word_counts[w] = word_counts.get(w, 0) + 1
sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:MAX_VOCAB - 3]
vocab = {'[PAD]': PAD_ID, '[UNK]': UNK_ID, '[CLS]': CLS_ID}
for word, _ in sorted_words: vocab[word] = len(vocab)
V = len(vocab)

def encode(texts):
    ids = np.full((len(texts), MAX_SEQLEN), PAD_ID, dtype=np.int64)
    mask = np.zeros((len(texts), MAX_SEQLEN), dtype=np.float32)
    for i, t in enumerate(texts):
        toks = t.split()[:MAX_SEQLEN - 1]
        ids[i, 0], mask[i, 0] = CLS_ID, 1.0
        for j, w in enumerate(toks):
            ids[i, j + 1] = vocab.get(w, UNK_ID)
            mask[i, j + 1] = 1.0
    return ids, mask

Xtr, Mtr = encode(tr_norm); Ytr = tr_df['label'].astype(int).values
Xv, Mv = encode(va_norm); Yv = va_df['label'].astype(int).values
Xte, Mte = encode(te_norm); Yte = te_df['label'].astype(int).values

class_counts = np.bincount(Ytr)
class_weights = torch.tensor(len(Ytr) / (NUM_CLASSES * class_counts), dtype=torch.float)

print(f"Vocab: {V:,} | Train: {len(Ytr):,} | Val: {len(Yv):,} | Test: {len(Yte):,}")
print(f"Class counts: Neg={class_counts[0]} Neu={class_counts[1]} Pos={class_counts[2]}")
print(f"Class weights: {class_weights.tolist()}")

Vocab: 30,000 | Train: 38,914 | Val: 8,980 | Test: 11,975
Class counts: Neg=7393 Neu=17849 Pos=13672
Class weights: [1.754542589187622, 0.7267260551452637, 0.94875168800354]


## 2. Dataset

In [12]:
class DS(Dataset):
    def __init__(self, ids, mask, labels):
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.mask = torch.tensor(mask, dtype=torch.float)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.ids[i], self.mask[i], self.labels[i]

train_ds, val_ds, test_ds = DS(Xtr, Mtr, Ytr), DS(Xv, Mv, Yv), DS(Xte, Mte, Yte)

if USE_OVERSAMPLING:
    sample_weights = [class_weights[l].item() for l in Ytr]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(Ytr), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)

val_loader = DataLoader(val_ds, batch_size=BATCH*2)
test_loader = DataLoader(test_ds, batch_size=BATCH*2)

print(f"Oversampling: {USE_OVERSAMPLING} | Batches: train={len(train_loader)} val={len(val_loader)} test={len(test_loader)}")

Batches: train=609 val=71 test=94


## 3. Model Architecture

```
Input (Token IDs) -> Embedding -> [Transformer Block x3] -> Triple Pooling -> Classifier -> 3 Classes
```

In [13]:
class MultiScaleConv(nn.Module):
    def __init__(self, h, drop=0.1):
        super().__init__()
        c = h // 4
        self.convs = nn.ModuleList([nn.Conv1d(h, c, k, padding=k//2) for k in [2,3,4,5]])
        self.bn, self.drop = nn.BatchNorm1d(h), nn.Dropout(drop)
    def forward(self, x):
        t = x.transpose(1, 2); L = x.size(1)
        return self.drop(F.gelu(self.bn(torch.cat([c(t)[:,:,:L] for c in self.convs], 1))).transpose(1, 2))

class MHAttn(nn.Module):
    def __init__(self, h, heads, drop=0.1):
        super().__init__()
        self.heads, self.d = heads, h // heads
        self.qkv, self.proj = nn.Linear(h, h*3), nn.Linear(h, h)
        self.drop, self.scale = nn.Dropout(drop), (h // heads) ** -0.5
    def forward(self, x, mask=None):
        B, L, D = x.shape
        qkv = self.qkv(x).reshape(B, L, 3, self.heads, self.d).permute(2, 0, 3, 1, 4)
        a = (qkv[0] @ qkv[1].transpose(-2, -1)) * self.scale
        if mask is not None: a = a.masked_fill(mask.unsqueeze(1).unsqueeze(2) == 0, float('-inf'))
        a = self.drop(F.softmax(a, dim=-1))
        return self.proj((a @ qkv[2]).transpose(1, 2).reshape(B, L, D))

class Block(nn.Module):
    def __init__(self, h, heads, ff, drop=0.1):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(h), nn.LayerNorm(h)
        self.attn, self.conv = MHAttn(h, heads, drop), MultiScaleConv(h, drop)
        self.ff = nn.Sequential(nn.Linear(h, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, h), nn.Dropout(drop))
    def forward(self, x, mask=None):
        n = self.ln1(x)
        return x + self.attn(n, mask) + self.conv(n) + self.ff(self.ln2(x))

class AttnPool(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(h, h//2), nn.Tanh(), nn.Linear(h//2, 1, bias=False))
    def forward(self, x, mask):
        s = self.net(x).squeeze(-1).masked_fill(mask == 0, float('-inf'))
        return (x * F.softmax(s, dim=-1).unsqueeze(-1)).sum(1)

class ElectraHybrid(nn.Module):
    def __init__(self, V, h, heads, layers, ff, n_cls, max_len, drop=0.2):
        super().__init__()
        self.tok_emb = nn.Embedding(V, h, padding_idx=PAD_ID)
        self.pos_emb = nn.Embedding(max_len, h)
        self.ln, self.drop = nn.LayerNorm(h), nn.Dropout(drop)
        self.layers = nn.ModuleList([Block(h, heads, ff, drop) for _ in range(layers)])
        self.pool = AttnPool(h)
        self.classifier = nn.Sequential(
            nn.Linear(h*3, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(drop), nn.Linear(h, n_cls)
        )
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight)
            elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, std=0.02)
            elif isinstance(m, nn.Conv1d): nn.init.kaiming_normal_(m.weight)
    def forward(self, ids, mask):
        pos = torch.arange(ids.size(1), device=ids.device).unsqueeze(0)
        x = self.drop(self.ln(self.tok_emb(ids) + self.pos_emb(pos)))
        for layer in self.layers: x = layer(x, mask)
        cls, att = x[:, 0], self.pool(x, mask)
        mx = x.masked_fill(mask.unsqueeze(-1) == 0, float('-inf')).max(1)[0]
        return self.classifier(torch.cat([cls, att, mx], -1))

print("Model defined!")

Model defined!


## 4. Training Setup

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ElectraHybrid(V, HIDDEN, NUM_HEADS, NUM_LAYERS, FF_DIM, NUM_CLASSES, MAX_SEQLEN, DROPOUT).to(device)
print(f"Device: {device} | Params: {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=0.01)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean() if self.reduction == 'mean' else focal.sum()

if USE_FOCAL:
    criterion = FocalLoss(alpha=class_weights.to(device), gamma=FOCAL_GAMMA)
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=LABEL_SMOOTH)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)

def get_lr(step):
    if step < warmup_steps: return PEAK_LR * step / max(1, warmup_steps)
    p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return MIN_LR + 0.5 * (PEAK_LR - MIN_LR) * (1 + math.cos(math.pi * p))

loss_name = f'FocalLoss(gamma={FOCAL_GAMMA})' if USE_FOCAL else f'CrossEntropy(smooth={LABEL_SMOOTH})'
print(f"Loss: {loss_name} + ClassWeights")
print(f"Total steps: {total_steps}, Warmup: {warmup_steps}")

writer = SummaryWriter(log_dir='runs/electra_hybrid')
writer.add_text('Config', f"HIDDEN={HIDDEN} HEADS={NUM_HEADS} LAYERS={NUM_LAYERS} DROPOUT={DROPOUT}")
writer.add_text('Loss', loss_name + ' + ClassWeights')

Device: cpu | Params: 4,477,315
Loss: CrossEntropy + LabelSmoothing(0.1) + ClassWeights
Total steps: 6090, Warmup: 609


## 5. Training Loop

In [15]:
best_acc, best_ep, no_imp = 0, 0, 0
hist = {'tl': [], 'vl': [], 'va': []}
t0, gs = time.time(), 0

for ep in range(EPOCHS):
    et = time.time()
    model.train(); tl = 0
    for ids, mask, lbl in train_loader:
        ids, mask, lbl = ids.to(device), mask.to(device), lbl.to(device)
        for g in optimizer.param_groups: g['lr'] = get_lr(gs)
        gs += 1
        optimizer.zero_grad()
        loss = criterion(model(ids, mask), lbl)
        if not torch.isnan(loss):
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tl += loss.item()
    tl /= len(train_loader)

    model.eval(); p, t, vl = [], [], 0
    with torch.no_grad():
        for ids, mask, lbl in val_loader:
            ids, mask, lbl = ids.to(device), mask.to(device), lbl.to(device)
            o = model(ids, mask)
            vl += criterion(o, lbl).item()
            p.extend(o.argmax(-1).cpu().numpy())
            t.extend(lbl.cpu().numpy())
    vl /= len(val_loader); va = accuracy_score(t, p)
    hist['tl'].append(tl); hist['vl'].append(vl); hist['va'].append(va)
    writer.add_scalar('Loss/train', tl, ep+1)
    writer.add_scalar('Loss/val', vl, ep+1)
    writer.add_scalar('Accuracy/val', va, ep+1)
    writer.add_scalar('LR', get_lr(gs), ep+1)
    print(f"Ep {ep+1:02d}/{EPOCHS} | {time.time()-et:.0f}s | tl={tl:.4f} vl={vl:.4f} va={va:.4f}")

    if va > best_acc:
        best_acc, best_ep, no_imp = va, ep+1, 0
        os.makedirs(os.path.join(PROJECT, 'checkpoints'), exist_ok=True)
        torch.save(model.state_dict(), os.path.join(PROJECT, 'checkpoints', 'electra_best.pt'))
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            print(f"Early stop at ep {ep+1}"); break

writer.close()
print(f"\nDone {((time.time()-t0)/60):.1f}min | Best val_acc={best_acc:.4f} at ep {best_ep}")
print(f"TensorBoard: run 'tensorboard --logdir=runs' to view")

Ep 01/10 | 358s | tl=1.1877 vl=0.9510 va=0.5501
Ep 02/10 | 364s | tl=0.8717 vl=0.8774 va=0.6441
Ep 03/10 | 288s | tl=0.7103 vl=0.9149 va=0.6144
Ep 04/10 | 327s | tl=0.5896 vl=1.0041 va=0.6353
Ep 05/10 | 280s | tl=0.5003 vl=1.0929 va=0.6285
Ep 06/10 | 300s | tl=0.4396 vl=1.1743 va=0.6184
Early stop at ep 6

Done 31.9min | Best val_acc=0.6441 at ep 2


## 6. Test Evaluation

In [16]:
model.load_state_dict(torch.load(os.path.join(PROJECT, 'checkpoints', 'electra_best.pt'), weights_only=True))
model.eval(); p, t = [], []
with torch.no_grad():
    for ids, mask, lbl in test_loader:
        ids, mask, lbl = ids.to(device), mask.to(device), lbl.to(device)
        p.extend(model(ids, mask).argmax(-1).cpu().numpy())
        t.extend(lbl.cpu().numpy())
acc = accuracy_score(t, p)
report = classification_report(t, p, target_names=['Neg','Neu','Pos'], output_dict=True)
cm = confusion_matrix(t, p)

# TensorBoard: log test results
writer = SummaryWriter(log_dir='runs/electra_hybrid')
writer.add_scalar('Test/accuracy', acc)
labels = ['Neg', 'Neu', 'Pos']
cm_data = [[labels[i], labels[j], int(cm[i][j])] for i in range(3) for j in range(3)]
writer.add_table('Confusion Matrix', cm_data, ['Actual', 'Predicted', 'Count'])
per_class = [[c, f"{report[c]['precision']:.4f}", f"{report[c]['recall']:.4f}", f"{report[c]['f1-score']:.4f}", str(report[c]['support'])] for c in labels]
writer.add_table('Per-Class Metrics', per_class, ['Class', 'Precision', 'Recall', 'F1', 'Support'])
writer.close()

print(f"{'='*55}")
print(f"  TEST ACCURACY: {acc:.4f} ({acc*100:.2f}%)")
print(f"{'='*55}")
print(classification_report(t, p, target_names=['Neg','Neu','Pos'], digits=4))
print(f"TensorBoard: run 'tensorboard --logdir=runs' to view")

  TEST ACCURACY: 0.6512 (65.12%)
              precision    recall  f1-score   support

         Neg     0.5599    0.6283    0.5921      2276
         Neu     0.7163    0.5716    0.6358      5492
         Pos     0.6408    0.7675    0.6985      4207

    accuracy                         0.6512     11975
   macro avg     0.6390    0.6558    0.6421     11975
weighted avg     0.6601    0.6512    0.6495     11975



## 7. Visualization

In [17]:
print('All results logged to TensorBoard!')
print('Run: tensorboard --logdir=runs')
print('Open: http://localhost:6006')
print()
print('Available dashboards:')
print('  - Loss/train, Loss/val  (per epoch)')
print('  - Accuracy/val          (per epoch)')
print('  - LR                    (per epoch)')
print('  - Confusion Matrix      (table)')
print('  - Per-Class Metrics     (table)')
print('  - Config & Loss info    (text)')

Plot saved: e:\transformer-main\results\electra_final.png


C:\Users\sinon\AppData\Local\Temp\ipykernel_3116\1500186279.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Summary

In [18]:
print(f"{'='*55}")
print(f"  FINAL RESULTS")
print(f"{'='*55}")
print(f"  Test Accuracy : {acc:.4f} ({acc*100:.2f}%)")
print(f"  Best Val Acc  : {best_acc:.4f} (ep {best_ep})")
print(f"  Config        : HIDDEN={HIDDEN} HEADS={NUM_HEADS} LAYERS={NUM_LAYERS}")
print(f"                  DROPOUT={DROPOUT} LR={PEAK_LR} EPOCHS={EPOCHS}")
print(f"  Loss          : CrossEntropy + LabelSmoothing({LABEL_SMOOTH}) + ClassWeights")
print(f"{'='*55}")
print(f"\nPer-class metrics:")
for c in ['Neg','Neu','Pos']:
    print(f"  {c:5s} | P={report[c]['precision']:.4f} R={report[c]['recall']:.4f} F1={report[c]['f1-score']:.4f}")
print(f"\nSaved: checkpoints/electra_best.pt")
print(f"Saved: results/electra_final.png")

  FINAL RESULTS
  Test Accuracy : 0.6512 (65.12%)
  Best Val Acc  : 0.6441 (ep 2)
  Config        : HIDDEN=128 HEADS=4 LAYERS=3
                  DROPOUT=0.15 LR=0.0005 EPOCHS=10
  Loss          : CrossEntropy + LabelSmoothing(0.1) + ClassWeights

Per-class metrics:
  Neg   | P=0.5599 R=0.6283 F1=0.5921
  Neu   | P=0.7163 R=0.5716 F1=0.6358
  Pos   | P=0.6408 R=0.7675 F1=0.6985

Saved: checkpoints/electra_best.pt
Saved: results/electra_final.png
